# Working hours

In [299]:
# employee id, meeting id, join time, leave time。算出来每个员工每天最长工作时长 （应该是去掉meeting的最大时长）
import pandas as pd
meetings = pd.DataFrame(
    [
        (2, 103, "2024-01-02 10:00:00", "2024-01-02 11:00:00"),
        (1, 101, "2024-01-01 09:00:00", "2024-01-01 10:00:00"),
        (3, 105, "2024-01-03 09:35:00", "2024-01-03 10:10:00"),
        (1, 104, "2024-01-02 14:00:00", "2024-01-02 15:30:00"),
        (2, 101, "2024-01-01 09:00:00", "2024-01-01 10:00:00"),
        (1, 107, "2024-01-04 13:00:00", "2024-01-04 14:00:00"),
        (3, 102, "2024-01-01 12:05:00", "2024-01-01 13:00:00"),
        (2, 108, "2024-01-05 15:10:00", "2024-01-05 16:00:00"),
        (1, 102, "2024-01-01 12:00:00", "2024-01-01 13:00:00"),
        (3, 107, "2024-01-04 13:00:00", "2024-01-04 14:00:00"),
        (2, 105, "2024-01-03 09:30:00", "2024-01-03 10:15:00"),
        (1, 108, "2024-01-05 15:00:00", "2024-01-05 16:00:00"),
        (3, 103, "2024-01-02 10:00:00", "2024-01-02 10:45:00"),
        (2, 106, "2024-01-04 11:00:00", "2024-01-04 12:00:00"),
        (1, 105, "2024-01-03 09:30:00", "2024-01-03 10:15:00"),
    ],
    columns=["employee_id", "meeting_id", "join_time", "leave_time"]
).assign(
    join_time=lambda x: pd.to_datetime(x["join_time"]),
    leave_time=lambda x: pd.to_datetime(x["leave_time"])
)

## Attempt 1

In [ ]:
df = meetings

In [ ]:
df['date'] = df['join_time'].dt.date

df.sort_values(by=['employee_id', 'date', 'join_time'], inplace=True)

df['day_start'] = pd.to_datetime(df['date'].astype(str) + " " + '09:00:00')
df['day_end'] = pd.to_datetime(df['date'].astype(str) + " " + '17:00:00')

In [ ]:
df

In [ ]:
working_tracker = {}
for i in range(len(df.index)):
    row = df.loc[df.index[i]]
    employee = row['employee_id']
    date_str = row['date'].strftime('%Y-%m-%d')
    leave_time = row['leave_time']
    if (employee, date_str) not in working_tracker:
        working_time_from_begin = (row['join_time'] - row['day_start']).seconds / 60
        working_time_to_end = (row['day_end'] - row['leave_time']).seconds / 60
        # print('working_time_from_begin: ', working_time_from_begin)
        # print('working_time_to_end: ', working_time_to_end)
        working_tracker[(employee, date_str)] = [leave_time, working_time_from_begin, working_time_to_end] # last meeting, longest time so far, longest time to end
    else:
        working_time = (row['join_time'] - working_tracker[(employee, date_str)][0]).seconds / 60
        working_tracker[(employee, date_str)][2] = (row['day_end'] - row['leave_time']).seconds / 60
        working_tracker[(employee, date_str)][1] = max( working_time, working_tracker[(employee, date_str)][1])
    

In [ ]:
working_tracker

In [ ]:
res = pd.DataFrame(columns=['employee_id','date','longest_work_duration_minutes'])

In [ ]:
print(len(res.columns))

In [ ]:
for i, key in enumerate(working_tracker):
    value = working_tracker[key]
    longest = max(value[1], value[2])
    res.loc[i] = [key[0], key[1], longest]
res['date'] = pd.to_datetime(res['date'])

In [ ]:
def longest_working_hour(df):
    df['date'] = df['join_time'].dt.date

    df.sort_values(by=['employee_id', 'date', 'join_time'], inplace=True)
    
    df['day_start'] = pd.to_datetime(df['date'].astype(str) + " " + '00:00:00')
    df['day_end'] = pd.to_datetime(df['date'].astype(str) + " " + '23:59:59')
    working_tracker = {}
    for i in range(len(df.index)):
        row = df.loc[df.index[i]]
        employee = row['employee_id']
        date_str = row['date'].strftime('%Y-%m-%d')
        leave_time = row['leave_time']
        if (employee, date_str) not in working_tracker:
            working_time_from_begin = (row['join_time'] - row['day_start']).total_seconds() / 60
            working_time_to_end = (row['day_end'] - row['leave_time']).total_seconds() / 60
            
            working_tracker[(employee, date_str)] = [leave_time, working_time_from_begin, working_time_to_end] # last meeting, longest time so far, longest time to end
        else:
            working_time = (row['join_time'] - working_tracker[(employee, date_str)][0]).total_seconds() / 60            
            working_tracker[(employee, date_str)][1] = max(working_time, working_tracker[(employee, date_str)][1])
            
            if row['leave_time'] > working_tracker[(employee, date_str)][0]:
                working_tracker[(employee, date_str)][2] = (row['day_end'] - row['leave_time']).total_seconds() / 60
                working_tracker[(employee, date_str)][0] = row['leave_time']
    
    res = pd.DataFrame(columns=['employee_id','date','longest_work_duration_minutes'])
    for i, key in enumerate(working_tracker):
        value = working_tracker[key]
        longest = max(value[1], value[2])
        res.loc[i] = [key[0], key[1], longest]
    res['date'] = pd.to_datetime(res['date'])
    return res
    

In [ ]:
res = longest_working_hour(df)

In [ ]:
res

## Attempt 2

In [ ]:
df = meetings

In [4]:
df

,employee_id,meeting_id,join_time,leave_time
0,2,103,2024-01-02 10:00:00,2024-01-02 11:00:00
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00
2,3,105,2024-01-03 09:35:00,2024-01-03 10:10:00
3,1,104,2024-01-02 14:00:00,2024-01-02 15:30:00
4,2,101,2024-01-01 09:00:00,2024-01-01 10:00:00
5,1,107,2024-01-04 13:00:00,2024-01-04 14:00:00
6,3,102,2024-01-01 12:05:00,2024-01-01 13:00:00
7,2,108,2024-01-05 15:10:00,2024-01-05 16:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00
9,3,107,2024-01-04 13:00:00,2024-01-04 14:00:00


In [ ]:
df['date'] = df['join_time'].dt.date

df = df.sort_values(['employee_id', 'join_time'])

df['day_start'] = pd.to_datetime(df['date'].astype(str) + ' 09:00:00')
df['day_end'] = pd.to_datetime(df['date'].astype(str) + ' 17:00:00')

rows = []
for (employee_id, date), g in df.groupby(['employee_id', 'date']):
    work_duration = get_work_duration(g)
    rows.append({'employee_id': employee_id, 'date': date, 'longest_work_duration': work_duration})

In [28]:
g

,employee_id,meeting_id,join_time,leave_time,date,day_start,day_end
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00,2024-01-01,2024-01-01 09:00:00,2024-01-01 17:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00,2024-01-01,2024-01-01 09:00:00,2024-01-01 17:00:00


In [32]:
tracker = [g.iloc[0]['day_start'], 0, (g.iloc[0]['day_end'] - g.iloc[0]['day_start']).total_seconds() / 60]
# 0: last leave time, 
# 1: max cont working duration till current meeting, 
# 2: working duration from current meeting to end of day

for i in range(len(g)):
    row = g.iloc[i]
    if row['join_time'] > tracker[0]:
        tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds()/60)
    if row['leave_time'] > tracker[0]:
        tracker[0] = row['leave_time']
        tracker[2] = (row['day_end'] - row['leave_time']).total_seconds() / 60
tracker

[Timestamp('2024-01-01 13:00:00'), 120.0, 240.0]

In [33]:
def get_work_duration(g):
    tracker = [g.iloc[0]['day_start'], 0, (g.iloc[0]['day_end'] - g.iloc[0]['day_start']).total_seconds() / 60]
    # 0: last leave time, 
    # 1: max cont working duration till current meeting, 
    # 2: working duration from current meeting to end of day
    
    for i in range(len(g)):
        row = g.iloc[i]
        if row['join_time'] > tracker[0]:
            tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds()/60)
        if row['leave_time'] > tracker[0]:
            tracker[0] = row['leave_time']
            tracker[2] = (row['day_end'] - row['leave_time']).total_seconds() / 60
    return max(tracker[1], tracker[2])

In [35]:
pd.DataFrame(rows)

,employee_id,date,longest_work_duration
0,1,2024-01-01,240.0
1,1,2024-01-02,300.0
2,1,2024-01-03,405.0
3,1,2024-01-04,240.0
4,1,2024-01-05,360.0
5,2,2024-01-01,420.0
6,2,2024-01-02,360.0
7,2,2024-01-03,405.0
8,2,2024-01-04,300.0
9,2,2024-01-05,370.0


## Attempt 3

In [90]:
meetings

,employee_id,meeting_id,join_time,leave_time
0,2,103,2024-01-02 10:00:00,2024-01-02 11:00:00
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00
2,3,105,2024-01-03 09:35:00,2024-01-03 10:10:00
3,1,104,2024-01-02 14:00:00,2024-01-02 15:30:00
4,2,101,2024-01-01 09:00:00,2024-01-01 10:00:00
5,1,107,2024-01-04 13:00:00,2024-01-04 14:00:00
6,3,102,2024-01-01 12:05:00,2024-01-01 13:00:00
7,2,108,2024-01-05 15:10:00,2024-01-05 16:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00
9,3,107,2024-01-04 13:00:00,2024-01-04 14:00:00


In [91]:
meetings.sort_values(['employee_id', 'join_time'], inplace= True)

In [92]:
meetings

,employee_id,meeting_id,join_time,leave_time
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00
3,1,104,2024-01-02 14:00:00,2024-01-02 15:30:00
14,1,105,2024-01-03 09:30:00,2024-01-03 10:15:00
5,1,107,2024-01-04 13:00:00,2024-01-04 14:00:00
11,1,108,2024-01-05 15:00:00,2024-01-05 16:00:00
4,2,101,2024-01-01 09:00:00,2024-01-01 10:00:00
0,2,103,2024-01-02 10:00:00,2024-01-02 11:00:00
10,2,105,2024-01-03 09:30:00,2024-01-03 10:15:00
13,2,106,2024-01-04 11:00:00,2024-01-04 12:00:00


In [95]:
meetings['date'] = pd.to_datetime(meetings['join_time'].dt.strftime('%Y-%m-%d'))

In [120]:
res = {}
for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
    # print('emp_id: ', emp_id)
    # print('date: ', date)
    # print('g:\n', g)
    # break
    longest_hour = get_longest_hour(g, date)
    res[(emp_id, date)] = longest_hour

In [105]:
day_start = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 09:00:00')
day_end = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 17:00:00')

In [117]:
tracker = [day_start, 0, (day_end - day_start).total_seconds() / 60]
# last leave time
# longest cont'd working hour till now
# longest cont'd working hour to end of day
for i in range(len(g)):
    row = g.iloc[i]
    tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds()/60)
    tracker[2] = min(tracker[2], (day_end - row['leave_time']).total_seconds()/60)
    tracker[0] = max(tracker[0], row['leave_time'])
longest_hour = max(tracker[1], tracker[2], 0)

In [118]:
longest_hour

240.0

In [119]:
def get_longest_hour(g, date):
    day_start = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 09:00:00')
    day_end = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 17:00:00')
    
    tracker = [day_start, 0, (day_end - day_start).total_seconds() / 60]
    # last leave time
    # longest cont'd working hour till now
    # longest cont'd working hour to end of day
    for i in range(len(g)):
        row = g.iloc[i]
        tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds()/60)
        tracker[2] = min(tracker[2], (day_end - row['leave_time']).total_seconds()/60)
        tracker[0] = max(tracker[0], row['leave_time'])
    longest_hour = max(tracker[1], tracker[2], 0)
    return longest_hour

In [121]:
res

{(1, Timestamp('2024-01-01 00:00:00')): 240.0,
 (1, Timestamp('2024-01-02 00:00:00')): 300.0,
 (1, Timestamp('2024-01-03 00:00:00')): 405.0,
 (1, Timestamp('2024-01-04 00:00:00')): 240.0,
 (1, Timestamp('2024-01-05 00:00:00')): 360.0,
 (2, Timestamp('2024-01-01 00:00:00')): 420.0,
 (2, Timestamp('2024-01-02 00:00:00')): 360.0,
 (2, Timestamp('2024-01-03 00:00:00')): 405.0,
 (2, Timestamp('2024-01-04 00:00:00')): 300.0,
 (2, Timestamp('2024-01-05 00:00:00')): 370.0,
 (3, Timestamp('2024-01-01 00:00:00')): 240.0,
 (3, Timestamp('2024-01-02 00:00:00')): 375.0,
 (3, Timestamp('2024-01-03 00:00:00')): 410.0,
 (3, Timestamp('2024-01-04 00:00:00')): 240.0}

In [122]:
rows = []
for key, value in res.items():
    emp_id, date = key
    rows.append({'employee_id': emp_id, 'date': date, 'longest_continuous_working_minutes': value})

In [124]:
pd.DataFrame(rows)

,employee_id,date,longest_continuous_working_minutes
0,1,2024-01-01,240.0
1,1,2024-01-02,300.0
2,1,2024-01-03,405.0
3,1,2024-01-04,240.0
4,1,2024-01-05,360.0
5,2,2024-01-01,420.0
6,2,2024-01-02,360.0
7,2,2024-01-03,405.0
8,2,2024-01-04,300.0
9,2,2024-01-05,370.0


## Attempt 4

In [233]:
meetings

,employee_id,meeting_id,join_time,leave_time
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00
3,1,104,2024-01-02 14:00:00,2024-01-02 15:30:00
14,1,105,2024-01-03 09:30:00,2024-01-03 10:15:00
5,1,107,2024-01-04 13:00:00,2024-01-04 14:00:00
11,1,108,2024-01-05 15:00:00,2024-01-05 16:00:00
4,2,101,2024-01-01 09:00:00,2024-01-01 10:00:00
0,2,103,2024-01-02 10:00:00,2024-01-02 11:00:00
10,2,105,2024-01-03 09:30:00,2024-01-03 10:15:00
13,2,106,2024-01-04 11:00:00,2024-01-04 12:00:00


In [230]:
meetings.sort_values(['employee_id', 'join_time'], inplace=True)

In [234]:
meetings['date'] = meetings['join_time'].dt.date

In [241]:
meetings['start_time'] = pd.to_datetime(meetings['date'].astype(str) + " 09:00:00")
meetings['end_time'] = pd.to_datetime(meetings['date'].astype(str) + " 17:00:00")

In [243]:
for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
    # print(g)
    # break
    

   employee_id  meeting_id           join_time          leave_time  \
1            1         101 2024-01-01 09:00:00 2024-01-01 10:00:00   
8            1         102 2024-01-01 12:00:00 2024-01-01 13:00:00   

         date          start_time            end_time  
1  2024-01-01 2024-01-01 09:00:00 2024-01-01 17:00:00  
8  2024-01-01 2024-01-01 09:00:00 2024-01-01 17:00:00  


In [254]:
start_time = g.loc[g.index[0], 'start_time']
end_time = g.loc[g.index[0], 'end_time']
tracker = [start_time, 0, (end_time-start_time).total_seconds() / 60]
# 0. last meeting end time
# 1. longest cont'd working time so far
# 2. longest cont'd time till end of the day

In [255]:
tracker

[Timestamp('2024-01-01 09:00:00'), 0, 480.0]

In [256]:
for i in range(len(g)):
    row = g.iloc[i]
    if row['join_time'] > tracker[0]:
        tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds() / 60)
    tracker[0] = max(tracker[0], row['leave_time'])
    tracker[2] = min(tracker[2], max(0, (end_time - row['leave_time']).total_seconds() / 60))



In [257]:
tracker

[Timestamp('2024-01-01 13:00:00'), 120.0, 240.0]

In [258]:
res = []
for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
    start_time = g.loc[g.index[0], 'start_time']
    end_time = g.loc[g.index[0], 'end_time']
    tracker = [start_time, 0, (end_time-start_time).total_seconds() / 60]
    for i in range(len(g)):
        row = g.iloc[i]
        if row['join_time'] > tracker[0]:
            tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds() / 60)
        tracker[0] = max(tracker[0], row['leave_time'])
        tracker[2] = min(tracker[2], max(0, (end_time - row['leave_time']).total_seconds() / 60))

    
    res.append({'employee_id': emp_id, 'date': date, 'longest_continuous_working_hour': max(tracker[1], tracker[2])})

    

In [260]:
pd.DataFrame(res)

,employee_id,date,longest_continuous_working_hour
0,1,2024-01-01,240.0
1,1,2024-01-02,300.0
2,1,2024-01-03,405.0
3,1,2024-01-04,240.0
4,1,2024-01-05,360.0
5,2,2024-01-01,420.0
6,2,2024-01-02,360.0
7,2,2024-01-03,405.0
8,2,2024-01-04,300.0
9,2,2024-01-05,370.0


## Attempt 5

In [304]:
meetings['date'] = meetings.join_time.dt.date

In [308]:
for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
    print('emp_id: ', emp_id)
    print('date: ', date)
    print(g)
    break

emp_id:  1
date:  2024-01-01
   employee_id  meeting_id           join_time          leave_time        date
1            1         101 2024-01-01 09:00:00 2024-01-01 10:00:00  2024-01-01
8            1         102 2024-01-01 12:00:00 2024-01-01 13:00:00  2024-01-01


In [314]:
start_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 09:00:00')
end_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 17:00:00')

In [315]:
tracker = [start_time, 0, (end_time - start_time).total_seconds() / 60]
# [0] last meeting end time
# [1] longest cont'd time so far
# [2] long cont'd time till end of day

In [318]:
for i in range(len(g)):
    row = g.iloc[i]
    if row['join_time'] > tracker[0]:
        tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds() / 60)
    tracker[2] = min(tracker[2], max(0, (end_time - row['leave_time']).total_seconds()/60))
    tracker[0] = max(tracker[0], row['leave_time'])
longest_hour = max(tracker[1], tracker[2], 0)

In [319]:
longest_hour

240.0

In [320]:
tracker

[Timestamp('2024-01-01 13:00:00'), 120.0, 240.0]

In [322]:
res = []
for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
    start_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 09:00:00')
    end_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 17:00:00')
    tracker = [start_time, 0, (end_time - start_time).total_seconds() / 60]
    # [0] last meeting end time
    # [1] longest cont'd time so far
    # [2] long cont'd time till end of day
    for i in range(len(g)):
        row = g.iloc[i]
        if row['join_time'] > tracker[0]:
            tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds() / 60)
        tracker[2] = min(tracker[2], max(0, (end_time - row['leave_time']).total_seconds()/60))
        tracker[0] = max(tracker[0], row['leave_time'])
    longest_minutes = max(tracker[1], tracker[2], 0)
    res.append({'employee_id': emp_id, 'date': date, 'longest_continuous_working_minutes': longest_minutes})

In [324]:
pd.DataFrame(res)

,employee_id,date,longest_continuous_working_minutes
0,1,2024-01-01,240.0
1,1,2024-01-02,300.0
2,1,2024-01-03,405.0
3,1,2024-01-04,240.0
4,1,2024-01-05,360.0
5,2,2024-01-01,420.0
6,2,2024-01-02,360.0
7,2,2024-01-03,405.0
8,2,2024-01-04,300.0
9,2,2024-01-05,370.0


In [328]:
def find_longest_cont_working_min(meetings):
    meetings['date'] = meetings.join_time.dt.date
    res = []
    for (emp_id, date), g in meetings.groupby(['employee_id', 'date']):
        start_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 09:00:00')
        end_time = pd.to_datetime(date.strftime('%Y-%m-%d') + ' 17:00:00')
        tracker = [start_time, 0, (end_time - start_time).total_seconds() / 60]
        # [0] last meeting end time
        # [1] longest cont'd time so far
        # [2] long cont'd time till end of day
        for i in range(len(g)):
            row = g.iloc[i]
            if row['join_time'] > tracker[0]:
                tracker[1] = max(tracker[1], (row['join_time'] - tracker[0]).total_seconds() / 60)
            tracker[2] = min(tracker[2], max(0, (end_time - row['leave_time']).total_seconds()/60))
            tracker[0] = max(tracker[0], row['leave_time'])
        longest_minutes = max(tracker[1], tracker[2], 0)
        res.append({'employee_id': emp_id, 'date': date, 'longest_continuous_working_minutes': longest_minutes})
    df = pd.DataFrame(res)
    return df

In [329]:
df = find_longest_cont_working_min(meetings)

In [330]:
df

,employee_id,date,longest_continuous_working_minutes
0,1,2024-01-01,240.0
1,1,2024-01-02,300.0
2,1,2024-01-03,405.0
3,1,2024-01-04,240.0
4,1,2024-01-05,360.0
5,2,2024-01-01,420.0
6,2,2024-01-02,360.0
7,2,2024-01-03,405.0
8,2,2024-01-04,300.0
9,2,2024-01-05,370.0


# Cohort based retention rate

In [261]:
activities = pd.DataFrame([
    (1, "2024-01-01"),
    (1, "2024-01-02"),
    (1, "2024-01-04"),

    (2, "2024-01-01"),
    (2, "2024-01-01"),
    (2, "2024-01-03"),

    (3, "2024-01-02"),
    (3, "2024-01-03"),
    (3, "2024-01-05"),

    (4, "2024-01-02"),

    (5, "2024-01-03"),
    (5, "2024-01-04"),

    (6, "2024-01-03"),
    (6, "2024-01-05"),
], columns=["user_id", "activity_date"])

activities["activity_date"] = pd.to_datetime(activities["activity_date"])

## Attempt 1

In [174]:
df = activities

In [175]:
df.drop_duplicates(['user_id', 'activity_date'], inplace=True)

In [176]:
df['cohort_date'] = df.groupby(by=['user_id'])['activity_date'].transform('min')

In [177]:
df['date_diff'] = (df['activity_date'] - df['cohort_date']).dt.days

In [178]:
df

,user_id,activity_date,cohort_date,diff,date_diff
0,1,2024-01-01,2024-01-01,0,0
1,1,2024-01-02,2024-01-01,1,1
2,1,2024-01-04,2024-01-01,3,3
3,2,2024-01-01,2024-01-01,0,0
5,2,2024-01-03,2024-01-01,2,2
6,3,2024-01-02,2024-01-02,0,0
7,3,2024-01-03,2024-01-02,1,1
8,3,2024-01-05,2024-01-02,3,3
9,4,2024-01-02,2024-01-02,0,0
10,5,2024-01-03,2024-01-03,0,0


In [179]:
retention = df.groupby(['cohort_date', 'date_diff'], as_index=False)['user_id'].nunique()

In [180]:
retention

,cohort_date,date_diff,user_id
0,2024-01-01,0,2
1,2024-01-01,1,1
2,2024-01-01,2,1
3,2024-01-01,3,1
4,2024-01-02,0,2
5,2024-01-02,1,1
6,2024-01-02,3,1
7,2024-01-03,0,2
8,2024-01-03,1,1
9,2024-01-03,2,1


In [181]:
cohort_size = df[df['date_diff'] == 0].groupby('cohort_date')['user_id'].nunique().reset_index(name='total_users')

In [182]:
res = retention.merge(cohort_size, on='cohort_date', how='inner')

In [183]:
res['retention_rate'] = res['user_id'] / res['total_users']

In [184]:
res.sort_values(by=['cohort_date', 'date_diff'])

,cohort_date,date_diff,user_id,total_users,retention_rate
0,2024-01-01,0,2,2,1.0
1,2024-01-01,1,1,2,0.5
2,2024-01-01,2,1,2,0.5
3,2024-01-01,3,1,2,0.5
4,2024-01-02,0,2,2,1.0
5,2024-01-02,1,1,2,0.5
6,2024-01-02,3,1,2,0.5
7,2024-01-03,0,2,2,1.0
8,2024-01-03,1,1,2,0.5
9,2024-01-03,2,1,2,0.5


In [185]:
def cohort_based_retention_rate(df):
    
    df.drop_duplicates(['user_id', 'activity_date'], inplace=True)
    df['cohort_date'] = df.groupby(['user_id'])['activity_date'].transform('min')
    df['date_diff'] = (df['activity_date'] - df['cohort_date']).dt.days

    # retention
    retention = df.groupby(['cohort_date', 'date_diff'], as_index=False)['user_id'].nunique()

    # user count per cohort
    user_count = df[df['date_diff'] == 0].groupby('cohort_date', as_index=False)['user_id'].nunique().rename(columns={'user_id':'total_users'})
    # merge
    res = retention.merge(user_count, on='cohort_date', how='inner')
    res['retention_rate'] = res['user_id'] / res['total_users']
    return res

In [186]:
res = cohort_based_retention_rate(df)

In [187]:
res

,cohort_date,date_diff,user_id,total_users,retention_rate
0,2024-01-01,0,2,2,1.0
1,2024-01-01,1,1,2,0.5
2,2024-01-01,2,1,2,0.5
3,2024-01-01,3,1,2,0.5
4,2024-01-02,0,2,2,1.0
5,2024-01-02,1,1,2,0.5
6,2024-01-02,3,1,2,0.5
7,2024-01-03,0,2,2,1.0
8,2024-01-03,1,1,2,0.5
9,2024-01-03,2,1,2,0.5


In [ ]:
df

In [ ]:
def get_min(g):
    return g.min()
df.groupby('user_id')[['activity_date']].transform(get_min)

## Attempt 2

In [161]:
activities

,user_id,activity_date
0,1,2024-01-01
1,1,2024-01-02
2,1,2024-01-04
3,2,2024-01-01
4,2,2024-01-01
5,2,2024-01-03
6,3,2024-01-02
7,3,2024-01-03
8,3,2024-01-05
9,4,2024-01-02


In [162]:
activities.drop_duplicates(['user_id', 'activity_date'], inplace=True)

In [163]:
activities['cohort_date'] = activities.groupby('user_id')['activity_date'].transform('min')

In [164]:
activities['diff'] = (activities['activity_date'] - activities['cohort_date']).dt.days

In [165]:
retention = activities.groupby(['cohort_date', 'diff'])['user_id'].nunique().reset_index().rename(columns={'user_id': 'id_counts'})

In [167]:
retention

,cohort_date,diff,id_counts
0,2024-01-01,0,2
1,2024-01-01,1,1
2,2024-01-01,2,1
3,2024-01-01,3,1
4,2024-01-02,0,2
5,2024-01-02,1,1
6,2024-01-02,3,1
7,2024-01-03,0,2
8,2024-01-03,1,1
9,2024-01-03,2,1


In [169]:
total_users = retention[retention['diff'] == 0][['cohort_date', 'id_counts']].rename(columns={'id_counts': 'total_ids'})

In [171]:
retention = retention.merge(total_users, on='cohort_date', how='inner')

In [172]:
retention['retention_rate'] = retention['id_counts'] / retention['total_ids']

In [173]:
retention

,cohort_date,diff,id_counts,total_ids,retention_rate
0,2024-01-01,0,2,2,1.0
1,2024-01-01,1,1,2,0.5
2,2024-01-01,2,1,2,0.5
3,2024-01-01,3,1,2,0.5
4,2024-01-02,0,2,2,1.0
5,2024-01-02,1,1,2,0.5
6,2024-01-02,3,1,2,0.5
7,2024-01-03,0,2,2,1.0
8,2024-01-03,1,1,2,0.5
9,2024-01-03,2,1,2,0.5


In [188]:
def get_retention_rate(activities):
    activities.drop_duplicates(['user_id', 'activity_date'], inplace=True)
    activities['cohort_date'] = activities.groupby('user_id')['activity_date'].transform('min')
    activities['diff'] = (activities['activity_date'] - activities['cohort_date']).dt.days
    retention = activities.groupby(['cohort_date', 'diff'])['user_id'].nunique().reset_index().rename(columns={'user_id': 'id_counts'})
    total_users = retention[retention['diff'] == 0][['cohort_date', 'id_counts']].rename(columns={'id_counts': 'total_ids'})
    retention = retention.merge(total_users, on='cohort_date', how='inner')
    retention['retention_rate'] = retention['id_counts'] / retention['total_ids']
    return retention

In [189]:
res = get_retention_rate(activities)

In [190]:
res

,cohort_date,diff,id_counts,total_ids,retention_rate
0,2024-01-01,0,2,2,1.0
1,2024-01-01,1,1,2,0.5
2,2024-01-01,2,1,2,0.5
3,2024-01-01,3,1,2,0.5
4,2024-01-02,0,2,2,1.0
5,2024-01-02,1,1,2,0.5
6,2024-01-02,3,1,2,0.5
7,2024-01-03,0,2,2,1.0
8,2024-01-03,1,1,2,0.5
9,2024-01-03,2,1,2,0.5


## Attempt 3

In [264]:
activities

,user_id,activity_date
0,1,2024-01-01
1,1,2024-01-02
2,1,2024-01-04
3,2,2024-01-01
4,2,2024-01-01
5,2,2024-01-03
6,3,2024-01-02
7,3,2024-01-03
8,3,2024-01-05
9,4,2024-01-02


In [265]:
activities.sort_values(['user_id', 'activity_date'], inplace=True)

In [ ]:
df.rename()

In [271]:
activities['cohort'] = activities.groupby('user_id')['activity_date'].transform('min')

In [273]:
activities['diff'] = (activities['activity_date'] - activities['cohort']).dt.days

In [279]:
activities

,user_id,activity_date,cohort,diff
0,1,2024-01-01,2024-01-01,0
1,1,2024-01-02,2024-01-01,1
2,1,2024-01-04,2024-01-01,3
3,2,2024-01-01,2024-01-01,0
4,2,2024-01-01,2024-01-01,0
5,2,2024-01-03,2024-01-01,2
6,3,2024-01-02,2024-01-02,0
7,3,2024-01-03,2024-01-02,1
8,3,2024-01-05,2024-01-02,3
9,4,2024-01-02,2024-01-02,0


In [291]:
retention = activities.groupby(['cohort', 'diff'])['user_id'].nunique().reset_index().rename(columns={'user_id': 'user_count'})

In [292]:
retention

,cohort,diff,user_count
0,2024-01-01,0,2
1,2024-01-01,1,1
2,2024-01-01,2,1
3,2024-01-01,3,1
4,2024-01-02,0,2
5,2024-01-02,1,1
6,2024-01-02,3,1
7,2024-01-03,0,2
8,2024-01-03,1,1
9,2024-01-03,2,1


In [293]:
total_users = activities.groupby(['cohort'])['user_id'].nunique().reset_index().rename(columns={'user_id':'total_users'})

In [296]:
res = retention.merge(total_users, how='left')

In [297]:
res['retention_rate'] = res['user_count'] / res['total_users']

In [298]:
res

,cohort,diff,user_count,total_users,retention_rate
0,2024-01-01,0,2,2,1.0
1,2024-01-01,1,1,2,0.5
2,2024-01-01,2,1,2,0.5
3,2024-01-01,3,1,2,0.5
4,2024-01-02,0,2,2,1.0
5,2024-01-02,1,1,2,0.5
6,2024-01-02,3,1,2,0.5
7,2024-01-03,0,2,2,1.0
8,2024-01-03,1,1,2,0.5
9,2024-01-03,2,1,2,0.5


# Another problem

In [ ]:
# 1. Given the link and company tables above, write a SQL query to count the number of link where the 
# firm company and the client company are operating in the same state.

# 2. Given the link table above, write a SQL query to report the average duration of a link in years. 
# Round the answer to 1 decimal place. Note that if the detach_date is null, then the link is currently active.

In [191]:
data = [
    [111, 123, "2018-01-10 17:25:41", None],
    [111, 234, "2016-03-08 16:47:12", None],
    [111, 345, "2015-12-10 18:01:46", None],
    [222, 456, "2019-03-31 18:07:47", "2022-03-29 12:51:58"],
    [222, 567, "2016-01-13 20:28:16", None],
    [333, 678, "2019-10-02 13:51:54", None],
    [333, 789, "2019-10-02 13:51:19", None],
    [333, 890, "2019-05-13 16:01:51", None],
    [333, 901, "2016-01-18 16:41:26", None],
]

link = pd.DataFrame(
    data,
    columns=[
        "firm_company_id",
        "client_company_id",
        "attach_date",
        "detach_date"
    ]
)

link["attach_date"] = pd.to_datetime(link["attach_date"])
link["detach_date"] = pd.to_datetime(link["detach_date"])

In [193]:
data = [
    [333, "2016-01-16 01:13:01", "2023-10-02 10:46:22", None],
    [222, "2016-01-13 17:35:40", "2019-03-31 18:22:38", None],
    [901, "2014-10-03 00:05:32", "2022-04-11 16:20:44", "HA"],
    [890, "2018-04-04 00:24:09", "2023-10-15 17:20:51", "NH"],
    [789, "2018-04-04 00:24:09", "2023-10-15 12:55:32", "NH"],
    [345, "2015-12-08 10:24:42", "2023-03-05 07:05:06", "IL"],
    [111, "2015-12-09 09:54:32", "2023-10-16 01:28:09", "IL"],
    [234, "2016-02-27 09:19:43", "2022-07-27 02:36:30", "IL"],
    [123, "2011-10-22 18:53:37", "2019-04-15 05:52:17", "IL"],
    [678, "2019-01-10 01:01:30", "2023-05-09 10:09:38", "LO"],
    [567, "2016-01-13 17:34:58", "2021-08-20 23:11:46", "MI"],
    [456, "2018-06-22 05:55:06", "2023-10-16 00:45:30", "VA"],
]

companies = pd.DataFrame(
    data,
    columns=[
        "company_id",
        "create_date",
        "last_accessed_date",
        "state"
    ]
)

companies["create_date"] = pd.to_datetime(companies["create_date"])
companies["last_accessed_date"] = pd.to_datetime(companies["last_accessed_date"])

In [200]:
states = companies.dropna(subset=['state'])[['company_id', 'state']]

In [204]:
res = link[['firm_company_id', 'client_company_id']].merge(states, left_on='firm_company_id', right_on='company_id', how='left')\
    .rename(columns={'state':'firm_state'}).merge(states, left_on='client_company_id', right_on='company_id', how='left').rename(columns={'state':'client_state'})

In [207]:
res = res.dropna(subset=['firm_state', 'client_state'])
res = res[res['firm_state'] == res['client_state']]

In [208]:
len(res)

3

In [209]:
def get_same_state(link, companies):
    states = companies.dropna(subset=['state'])[['company_id', 'state']]
    res = link[['firm_company_id', 'client_company_id']].merge(states, left_on='firm_company_id', right_on='company_id', how='left')\
    .rename(columns={'state':'firm_state'}).merge(states, left_on='client_company_id', right_on='company_id', how='left').rename(columns={'state':'client_state'})
    res = res.dropna(subset=['firm_state', 'client_state'])
    res = res[res['firm_state'] == res['client_state']]
    return len(res)

In [210]:
result = get_same_state(link, companies)

In [211]:
result

3

In [212]:
today = pd.Timestamp.today().normalize()

In [213]:
today

Timestamp('2026-05-01 00:00:00')

In [216]:
link.fillna(today, inplace=True)

In [221]:
duration = (link['detach_date'] - link['attach_date']).dt.days

In [222]:
duration.mean()

2943.0

In [223]:
def get_avg_duration(link):
    today = pd.Timestamp.today().normalize()
    link.fillna(today, inplace=True)
    duration = (link['detach_date'] - link['attach_date']).dt.days
    return duration.mean()

In [225]:
res = get_avg_duration(link)

In [226]:
res

2943.0

In [ ]:
## Other problems

In [36]:
data = [[1, 100], [2, 200], [3, 300]]
employee = pd.DataFrame(data, columns=['id', 'salary']).astype({'id':'int64', 'salary':'int64'})

In [43]:
sorted_salary = employee.sort_values(['salary'], ascending=False)['salary'].unique()

In [46]:
if len(sorted_salary) > 2:
    res = sorted_salary[1]
else:
    res = np.nan

pd.DataFrame({'SecondHighestSalary':[res]})

,SecondHighestSalary
0,200


In [51]:
data = [[1, 'ATGCTAGCTAGCTAA', 'Human'], [2, 'GGGTCAATCATC', 'Human'], [3, 'ATATATCGTAGCTA', 'Human'], [4, 'ATGGGGTCATCATAA', 'Mouse'], [5, 'TCAGTCAGTCAG', 'Mouse'], [6, 'ATATCGCGCTAG', 'Zebrafish'], [7, 'CGTATGCGTCGTA', 'Zebrafish']]
samples = pd.DataFrame(data,columns={
    'sample_id': pd.Series(dtype='int'),        # Equivalent to SERIAL/INTEGER
    'dna_sequence': pd.Series(dtype='string'),  # Equivalent to TEXT/VARCHAR
    'species': pd.Series(dtype='string')        # Equivalent to VARCHAR(100)
})

In [52]:
samples

,sample_id,dna_sequence,species
0,1,ATGCTAGCTAGCTAA,Human
1,2,GGGTCAATCATC,Human
2,3,ATATATCGTAGCTA,Human
3,4,ATGGGGTCATCATAA,Mouse
4,5,TCAGTCAGTCAG,Mouse
5,6,ATATCGCGCTAG,Zebrafish
6,7,CGTATGCGTCGTA,Zebrafish


In [53]:
samples['has_start'] = 0
samples['has_stop'] = 0
samples['has_atat'] = 0
samples['has_ggg'] = 0

In [55]:
a = 'aabbaa'
b = 'ab'
b in a

True

In [58]:
def seq_check(seq):
    has_start = int(seq[:3] == 'ATG')
    has_stop = int(seq[-3:] in ['TAA', 'TGA', 'TAG'])
    has_atat = int('ATAT' in seq)
    has_ggg = int('GGG' in seq)
    return has_start, has_stop, has_atat, has_ggg

In [59]:
for i in range(len(samples)):
    row = samples.iloc[i]
    has_start, has_stop, has_atat, has_ggg = seq_check(row['dna_sequence'])
    samples.loc[samples.index[i], 'has_start'] = has_start
    samples.loc[samples.index[i], 'has_stop'] = has_stop
    samples.loc[samples.index[i], 'has_atat'] = has_atat
    samples.loc[samples.index[i], 'has_ggg'] = has_ggg

In [60]:
samples

,sample_id,dna_sequence,species,has_start,has_stop,has_atat,has_ggg
0,1,ATGCTAGCTAGCTAA,Human,1,1,0,0
1,2,GGGTCAATCATC,Human,0,0,0,1
2,3,ATATATCGTAGCTA,Human,0,0,1,0
3,4,ATGGGGTCATCATAA,Mouse,1,1,0,1
4,5,TCAGTCAGTCAG,Mouse,0,0,0,0
5,6,ATATCGCGCTAG,Zebrafish,0,1,1,0
6,7,CGTATGCGTCGTA,Zebrafish,0,0,0,0


In [61]:
data = [[1, '2023-01-01', 'free_trial', 45], [1, '2023-01-02', 'free_trial', 30], [1, '2023-01-05', 'free_trial', 60], [1, '2023-01-10', 'paid', 75], [1, '2023-01-12', 'paid', 90], [1, '2023-01-15', 'paid', 65], [2, '2023-02-01', 'free_trial', 55], [2, '2023-02-03', 'free_trial', 25], [2, '2023-02-07', 'free_trial', 50], [2, '2023-02-10', 'cancelled', 0], [3, '2023-03-05', 'free_trial', 70], [3, '2023-03-06', 'free_trial', 60], [3, '2023-03-08', 'free_trial', 80], [3, '2023-03-12', 'paid', 50], [3, '2023-03-15', 'paid', 55], [3, '2023-03-20', 'paid', 85], [4, '2023-04-01', 'free_trial', 40], [4, '2023-04-03', 'free_trial', 35], [4, '2023-04-05', 'paid', 45], [4, '2023-04-07', 'cancelled', 0]]
user_activity = pd.DataFrame(data, columns={
    'user_id': pd.Series(dtype='int'),
    'activity_date': pd.Series(dtype='datetime64[ns]'),
    'activity_type': pd.Series(dtype='str'),
    'activity_duration': pd.Series(dtype='int')
})

In [64]:
paid_user = user_activity[user_activity['activity_type'] == 'paid'].drop_duplicates(['user_id'])['user_id'].to_list()

In [65]:
paid_user

[1, 3, 4]

In [71]:
res = user_activity[user_activity['user_id'].isin(paid_user) & (user_activity['activity_type']!= 'cancelled')]\
    .groupby(['user_id', 'activity_type'])['activity_duration'].mean().reset_index()

In [73]:
res

,user_id,activity_type,activity_duration
0,1,free_trial,45.000000
1,1,paid,76.666667
2,3,free_trial,70.000000
3,3,paid,63.333333
4,4,free_trial,37.500000
5,4,paid,45.000000


In [82]:
pivoted = res.pivot(
    index='user_id',
    columns='activity_type',
    values='activity_duration'
         ).reset_index()

In [85]:
pivoted['free_trial'] = round(pivoted['free_trial'], 2)
pivoted['paid'] = round(pivoted['paid'], 2)

In [86]:
pivoted

activity_type,user_id,free_trial,paid
0,1,45.0,76.67
1,3,70.0,63.33
2,4,37.5,45.00
